## Overview
In this notebook, we will run DCON on a Solovev ideal example equilibrium and plot the results

In [ ]:
# Load in necessary packages
using Pkg
using HDF5
using Plots
using LaTeXStrings

## Run the code
We will run the main DCON code using the inputs specified in `dcon.toml`, `equil.toml`, and `vac.in`. We run the code 3 times: once with both n=1 and n=2 at the same time using the files in this folder, and then for a single-n to compare the outputs with in their respective subfolders. Each run will output a `euler.h5` file (but we set the default file naming in the `dcon.toml` to contain an "_n1" and "_n2" for the single-n cases), which is a Julia version of the `euler.bin` file.

In [ ]:
# Run DCON in Julia
Pkg.activate("../..")
using JPEC
JPEC.main(["./"]) # "./" tells us to obtain inputs and direct outputs to our current folder
JPEC.main(["./single_n_1"])
JPEC.main(["./single_n_2"])

## Analyze Outputs
We will now analyze the outputs of the run, the most important of which are located in the `euler.h5` output file. We load in the data from both single-n runs and the multi-n run here.

In [ ]:
# Read in the multi-n run euler.h5 data
eh5 = h5open("euler.h5", "r")
mlow = read(eh5["info/mlow"])
mhigh = read(eh5["info/mhigh"])
nlow = read(eh5["info/nlow"])
psio = read(eh5["equil/psio"])
xi_psi = read(eh5["integration/xi_psi"])
psifac = read(eh5["integration/psi"])
wt = read(eh5["vacuum/wt"])
et = read(eh5["vacuum/et"])
mn_index = read(eh5["info/mn_index"])
close(eh5)

# scale energy eigenvector matrices
chi1 = 2π*psio
wt .*= (chi1*1e-3)
println("Done reading euler.h5")

In [ ]:
# Read in the single-n run with n=1 euler.h5 data
eh5 = h5open("single_n_1/euler_n1.h5", "r")
mlow = read(eh5["info/mlow"])
psio = read(eh5["equil/psio"])
xi_psi_n1 = read(eh5["integration/xi_psi"])
psifac_n1 = read(eh5["integration/psi"])
wt_n1 = read(eh5["vacuum/wt"])
mn_index_n1 = read(eh5["info/mn_index"])
close(eh5)

In [ ]:
# Read in the single-n run with n=2 euler.h5 data
eh5 = h5open("single_n_2/euler_n2.h5", "r")
mlow = read(eh5["info/mlow"])
psio = read(eh5["equil/psio"])
xi_psi_n2 = read(eh5["integration/xi_psi"])
psifac_n2 = read(eh5["integration/psi"])
mn_index_n2 = read(eh5["info/mn_index"])
close(eh5)

### Plot comparison of $\xi_{\psi}$ for a few poloidal mode numbers
For clarity, we dump the `mn_index` in the output HDF5 file to make it clear which component in the first dimension of $\xi_{\psi}$ is being extracted. We show here how to use it to extract a desired $(m,n)$ mode. Note that in the multi-n case, the least stable mode corresponds to an n=1 perturbation and the second least stable mode to an n=2 perturbation. Therefore, when comparing the n=1 components we look at the first column (the least stable mode) of both runs; however, when comparing n=2 we must look at the second column of the multi-n run which is identical to the least stable mode computed in the n=2 only run.

In [ ]:
p = plot()
n = 1
for m in 1:2
    plot!(psifac, imag.(xi_psi[findfirst(x -> x[1] == m && x[2] == n, eachrow(mn_index)), 1, :]), linewidth = 3, label="m=$m, n=$n, multi-n")
    plot!(psifac_n1, imag.(xi_psi_n1[findfirst(x -> x[1] == m && x[2] == n, eachrow(mn_index_n1)), 1, :]), linestyle = :dash, linewidth = 3, label="m=$m, n=$n single-n")
end
n = 2
for m in 1:2
    plot!(psifac, imag.(xi_psi[findfirst(x -> x[1] == m && x[2] == n, eachrow(mn_index)), 2, :]), linewidth = 3, label="m=$m, n=$n, multi-n")
    plot!(psifac_n2, imag.(xi_psi_n2[findfirst(x -> x[1] == m && x[2] == n, eachrow(mn_index_n2)), 1, :]), linestyle = :dash, linewidth = 3, label="m=$m, n=$n single-n")
end
xlabel!(L"\psi_N")
ylabel!(L"\mathrm{Im}(\xi_\psi)")
title!("Least Stable Eigenmode " * L"\xi_\psi")
display(p)

### Compare the eigenvectors and eigenvalues of each DCON energy matrix eigenmode
This is analagous to the DCON summary plot creating by OMFIT GPEC

In [ ]:
# I got tired of trying to get the Plots version of this to work, so here's a PyPlot version
using PyPlot

# Axes labels
xlabel = "m"
ylabel = "mode (least to most stable)"

yvals = 1:size(wt, 2)
xvals = 1:size(wt, 1)

# Compute corresponding m and n
mpert = mhigh - mlow + 1
m_vals = [(i - 1) % mpert + mlow for i in xvals]
n_vals = [(i - 1) ÷ mpert + nlow for i in xvals]

# Create figure and grid layout
fig = figure(figsize=(9, 7))
gs = fig.add_gridspec(2, 3, height_ratios=[0.25, 0.75], width_ratios=[0.75, 0.21, 0.04])

# Top-left: Eigenvector amplitude
ax0 = fig.add_subplot(gs[1, 1])
ax0.plot(xvals, abs.(wt[:, 1]), color="blue", marker="o", markersize=3)
ax0.set_ylabel("|Eigenvector|")
ax0.set_xlabel("")
ax0.set_xticks([])
ax0.set_title("Mode 1, eigenvalue = $(round(abs(et[1]), digits=3))")

# Bottom-left: Heatmap
ax1 = fig.add_subplot(gs[2, 1])
im = ax1.imshow(abs.(wt') , aspect="auto", origin="lower",
                cmap="viridis", extent=[xvals[1], xvals[end], yvals[1], yvals[end]])
ax1.set_xlabel(xlabel)
ax1.set_ylabel(ylabel)

# ===== UPPER X-AXIS: m =====
step_m = max(1, Int(length(xvals) ÷ 10))
xticks_m = xvals[1:step_m:end]
ax1.set_xticks(xticks_m)
ax1.set_xticklabels(["$(m_vals[i])" for i in 1:step_m:length(xvals)])
ax1.set_xlabel("m")

# ===== LOWER X-AXIS: n =====
# Create secondary axis *below* the first one
ax_n = ax1.secondary_xaxis("bottom", functions=(x->x, x->x))
ax_n.xaxis.set_label_position("bottom")
ax_n.xaxis.set_ticks_position("bottom")

# Position slightly below the original
ax_n.spines["bottom"].set_position(("outward", 40))

# Show only one n per mpert group
n_indices = [i for i in mpert÷2:mpert:length(xvals)]
xticks_n = xvals[n_indices]
xticklabels_n = ["$(n_vals[i])" for i in n_indices]

ax_n.set_xticks(xticks_n)
ax_n.set_xticklabels(xticklabels_n)
ax_n.set_xlabel("n")

# Right middle: Eigenvalue amplitude
ax2 = fig.add_subplot(gs[2, 2])
ax2.scatter(abs.(et), yvals, c="red", s=30)
ax2.set_xlabel("Eigenvalue")
ax2.set_xscale("log")
ax2.set_xlim(0.1 * minimum(abs.(et)), 10 * maximum(abs.(et)))
ax2.set_yticks([])

# Colorbar (bottom right)
cax = fig.add_subplot(gs[2, 3])
cb = fig.colorbar(im, cax=cax)
cb.set_label("|Wₜ|")

display(fig)

### Plot crit (the smallest eigenvalue of $W^{-1}$) versus $\Psi$
If crit changes signs during integration, we know we are unstable to an ideal fixed-boundary instability. It has the same interpretation when running with multiple toroidal modes, since the number of eigenvalues double but we still only need to monitor the smallest in magnitude. This data is saved in the `crit.h5` file.

In [ ]:
# Read in the crit.h5 data
crit_h5 = h5open("crit.h5", "r")
psi = read(crit_h5["psi"])
crit = read(crit_h5["crit"])
close(crit_h5)

# Plot crit vs psi
p = plot(psi, crit, legend=false)
xlabel!(L"\psi_N")
ylabel!(L"crit")
title!("Smallest eigenvalue of " * L"W^{-1}" * "(crit) versus " * L"\psi_N")